# AKT26 Session 4: GPU validation
Team **Sayf AI Ciao**, ID **1142**. Runs the repository's CUDA executables, not Python replacements.

In Colab select **Runtime > Change runtime type > GPU**, then **Runtime > Run all**.
A free GPU allocation is sufficient for these sizes; availability depends on Colab.
The notebook starts **unexecuted**. A saved copy with outputs and the downloaded ZIP are the evidence of a real run.

The organizers have authorized Colab as an alternative (team-provided information).
AutoHPC credentials have not been issued to this team. The PDF sets no fixed benchmark sweep.
The cases below cover both assignments and compare against cuBLAS in FP32.


In [ ]:
from pathlib import Path
import csv
import datetime
import json
import re
import shutil
import subprocess

REPOSITORY = "https://github.com/mbzuaiciao/akt26_gpu_dev.git"
REVISION = "session-4-assignments"  # For exact reruns, replace with the recorded commit SHA.
ROOT = Path("/content/akt26_gpu_dev")
RESULTS = Path("/content") / ("session4_results_" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
RESULTS.mkdir()

def run(args, log, cwd=None):
    result = subprocess.run([str(a) for a in args], cwd=cwd, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    text = "$ " + " ".join(str(a) for a in args) + "\n" + result.stdout
    (RESULTS / log).write_text(text + f"\nexit_code={result.returncode}\n")
    print(text)
    result.check_returncode()
    return result.stdout

assert shutil.which("nvidia-smi"), "Select a GPU runtime, then rerun."
assert shutil.which("nvcc"), "CUDA compiler nvcc is missing from this runtime."
run(["nvidia-smi"], "nvidia-smi.txt")
capability = run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], "compute-capability.txt").splitlines()[0].strip()
SM = capability.replace(".", "")
assert SM.isdigit(), f"Unrecognized compute capability: {capability}"
run(["nvcc", "--version"], "nvcc.txt")
run(["c++", "--version"], "host-compiler.txt")
run(["uname", "-a"], "host.txt")
assert not ROOT.exists(), "Repository directory already exists. Use a fresh Colab runtime to avoid overwriting work."
run(["git", "clone", REPOSITORY, ROOT], "clone.txt")
run(["git", "checkout", REVISION], "checkout.txt", ROOT)
COMMIT = run(["git", "rev-parse", "HEAD"], "commit.txt", ROOT).strip()
CUDA_DIR = Path(shutil.which("nvcc")).resolve().parent.parent
(RESULTS / "run.json").write_text(json.dumps({
    "team": "Sayf AI Ciao", "team_id": 1142, "repository": REPOSITORY,
    "requested_revision": REVISION, "commit": COMMIT, "sm": SM,
    "cuda_directory": str(CUDA_DIR),
    "nvcc_flags": "-O3 -std=c++11 -lineinfo", "platform": "Google Colab"
}, indent=2))


## Build with the supplied Makefiles
The detected GPU compute capability selects `sm`; the Makefiles also include PTX.
Build logs preserve the complete compiler command. Any failure stops the cell.


In [ ]:
for directory in ["3-ger", "4-gemm"]:
    run(["make", "clean"], directory + "-clean.txt", ROOT / directory)
    run(["make", f"sm={SM}", f"CUDADIR={CUDA_DIR}"], directory + "-build.txt", ROOT / directory)


## Correctness and boundary cases
Each executable compares custom CUDA and cuBLAS results with the supplied CPU reference, using the supplied `1e-6` tolerance.
GER uses one restored update for correctness. GEMM covers every supported size, including partial blocks, and both single and multiple matrices.
The existing CLI must reject invalid sizes. Compute Sanitizer additionally checks GPU memory access and GEMM synchronization when installed.


In [ ]:
def check(executable, args, label):
    output = run([ROOT / executable, *args], label + ".txt", ROOT)
    assert "TEST PASS" in output and "TEST FAIL" not in output, label
    return output

for m, n in [(1, 1), (1, 33), (33, 1), (31, 7), (32, 8), (33, 9), (257, 129)]:
    check("3-ger/ger", [m, n, 10], f"correctness-ger-{m}-{n}")
for n in range(1, 33):
    for batch in [1, 3]:
        check("4-gemm/gemm", [n, batch], f"correctness-gemm-{n}-{batch}")

for executable, args in [("3-ger/ger", [0, 1, 1]), ("3-ger/ger", [1, 0, 1]),
                         ("3-ger/ger", [1, 1, 0]), ("4-gemm/gemm", [0, 1]),
                         ("4-gemm/gemm", [33, 1]), ("4-gemm/gemm", [1, 0])]:
    result = subprocess.run([str(ROOT / executable), *map(str, args)], text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    with (RESULTS / "invalid-inputs.txt").open("a") as log:
        log.write(f"{executable} {args}: exit_code={result.returncode}\n{result.stdout}\n")
    assert result.returncode != 0 and "Usage:" in result.stdout, (executable, args)

if shutil.which("compute-sanitizer"):
    for tool, executable, args in [
        ("memcheck", "3-ger/ger", [33, 9, 2]),
        ("memcheck", "4-gemm/gemm", [31, 3]),
        ("racecheck", "4-gemm/gemm", [31, 3]),
        ("synccheck", "4-gemm/gemm", [31, 3]),
    ]:
        run(["compute-sanitizer", "--tool", tool, "--error-exitcode", "99",
             ROOT / executable, *args], f"sanitizer-{tool}-{Path(executable).name}.txt", ROOT)
else:
    (RESULTS / "sanitizer-unavailable.txt").write_text("compute-sanitizer not installed; GPU sanitizer checks NOT RUN.\n")
    print("compute-sanitizer unavailable; GPU sanitizer checks NOT RUN.")
(RESULTS / "correctness-passed.txt").write_text("All executable comparisons and invalid-input checks passed. See individual logs.\n")


## Benchmarks
Three independent runs per case, each with an untimed warmup and 100 timed repetitions.
Event timings exclude allocation and host/device transfers. GER timed repetitions accumulate updates; its accuracy check restores the input separately.
GEMM overwrites C on every call, with cuBLAS alpha=1, beta=0 and pedantic FP32 math.
Save all trials; compare timings only on the same allocated GPU. GFLOP/s follows the starter's operation counts (`2mn` for GER, `2n^3 S` for GEMM).


In [ ]:
rows = []
cases = [("ger", m, n) for m, n in [(256, 256), (1024, 1024), (4000, 2000)]]
cases += [("gemm", n, batch) for n, batch in [(8, 1000), (16, 1000), (32, 1000), (32, 10000)]]
for assignment, first, second in cases:
    for trial in range(1, 4):
        executable = "3-ger/ger" if assignment == "ger" else "4-gemm/gemm"
        args = [first, second, 100] if assignment == "ger" else [first, second]
        output = check(executable, args, f"benchmark-{assignment}-{first}-{second}-{trial}")
        for implementation in ["Custom kernel", "cuBLAS SGER" if assignment == "ger" else "cuBLAS Strided Batched"]:
            match = re.search(r"^" + re.escape(implementation) + r"\s+([0-9.eE+-]+)\s+([0-9.eE+-]+)\s+([0-9.eE+-]+)\s*$", output, re.MULTILINE)
            assert match, (implementation, output)
            us, gflops, error = map(float, match.groups())
            rows.append(dict(assignment=assignment, m_or_n=first, n_or_batch=second,
                             trial=trial, repeats=100, implementation=implementation,
                             time_us=us, gflops=gflops, error=error, commit=COMMIT, sm=SM))
        # Write incrementally so earlier trials survive a later failure.
        with (RESULTS / "benchmarks.csv").open("w", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)
(RESULTS / "benchmarks-completed.txt").write_text("All benchmark cases passed and all three trials were recorded.\n")
print((RESULTS / "benchmarks.csv").read_text())


## Preserve results
Run this cell even after a failed test to preserve diagnostics. Download the ZIP before disconnecting; `/content` is temporary.
Also choose **File > Save a copy in Drive** to retain cell outputs, or **File > Download > Download .ipynb** after execution.
Add the ZIP (or its logs and CSV) to the assignment branch and update `SESSION4_WALKTHROUGH.md` with actual results.
A complete run includes `correctness-passed.txt` and `benchmarks-completed.txt`; inspect sanitizer availability and logs separately.
Review numerical errors, GPU/compiler metadata, and speedups before submission. No organizer submission is performed here.


In [ ]:
from google.colab import files
archive = shutil.make_archive(str(RESULTS), "zip", RESULTS)
print("Results:", archive)
files.download(archive)
